deployment

load model pipeline, threshold, and features

In [1]:
import joblib
import pandas as pd
from feature_engineering import EngineerFeatures

package = joblib.load("readmission_production.joblib")

model = package["model"] # calibrated pipeline
threshold = package["threshold"]
features = package["features"]

option A: deployment via batch scoring

In [ ]:
def score_patients(df_new: pd.DataFrame):
    # ensure column order
    df_new = df_new[features]

    # predicted probability
    proba = model.predict_proba(df_new)[:,1]

    # risk classification
    pred = (proba >= threshold).astype(int)

    return pd.DataFrame({
        "readmission_risk": proba,
        "high_risk_flag": pred
    })

new_data = pd.read_csv("today_dischrages.csv")
scores = score_patients(new_data)
scores.to_csv("risk_scores.csv", index=False)

option B: REST API - live prediction service

In [ ]:
from fastapi import FastAPI

app = FastAPI() # creates application object

@app.post("/predict") # route decorator
def predict(patient: dict):
    df = pd.DataFrame([patient]) # convert input into data frame
    df = df[features] # enforce feature order

    proba = model.predict_proba(df)[:,1][0]
    flag = int(proba >= threshold)

    return {
        "readmission_risk": float(proba),
        "high_risk": flag
    }

# uvicorn app:app --host 0.0.0.0 --port 8000

# note: expectes request body to be JSON:
{
  "age": 72,
  "gender": "male",
  "length_of_stay_days": 5,
  ...
}